In [4]:
#!pip install -r requirements_coursera.txt
%pip install pyarrow fastparquet
%pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 63.8 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 130.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import glob
import numpy as np
import pandas as pd

In [6]:
class softmaxlogisticreg:

  def __init__(self, learning_rate=0.01, max_iter=500):
      self.lr = learning_rate
      self.max_iter = max_iter
      self.W = None
      self.classes = None


  def _softmax(self,A):
      exp_A = np.exp(A - np.max(A,axis = 1, keepdims = True))
      return exp_A / np.sum(exp_A,axis = 1, keepdims = True)

  def one_hot_encode(self,y,num_classes):
      return np.eye(num_classes)[y.astype(int)]


  def fit(self, X, y):
      #flatten label
      y=np.ravel(y).astype(int)
      self.classes = np.unique(y)
      num_classes = len(self.classes)
      n_samples, n_features = X.shape 

      #Map class labels
      label_to_idx = {c: i for i, c in enumerate(self.classes)}
      y_indexed = np.array([label_to_idx[val] for val in y])
      T = self.one_hot_encode(y_indexed, num_classes)

      #Form the design Matrix
      X_bias = np.hstack((np.ones((n_samples,1)),X))

      #Initialize weight vector
      self.W = np.random.random((n_features + 1, num_classes)) * 0.01

      #Gradient descent
      for i in range (self.max_iter):
      #compute score and convert to prob distr
          A = np.dot(X_bias,self.W)
          P = self._softmax(A)

      #compute cross entopy loss
          loss = -np.mean(np.sum(T*np.log(P+1e-15),axis =1))

      #compute the gradient matrix
          gradient = np.dot(X_bias.T,(P-T))/n_samples

    
      #shift weight
          self.W -= self.lr * gradient

          if i%50 ==  0:
              print(f"iteration {i} | Cross Entropy loss {loss}")

      return self

       


  def predict_proba(self, X):
      #append bias column to inputs
      X_bias = np.hstack((np.ones((len(X),1)),X))
      #compute score for probability
      return self._softmax(np.dot(X_bias, self.W))

  def predict(self, X):
      #Pick highest probability
      prob_dist = self.predict_proba(X) #accepts X with shape n_samples and n_features
      best_indices = np.argmax(prob_dist, axis=1)
      return self.classes[best_indices]
   

  def score(self, X, y): #calculate overall accuracy percentageZ
      y_pred = self.predict(X)
      y_true = np.ravel(y)
      return np.mean(y_pred == y_true)

      

In [7]:
#==============================================
#DATA LOADING AND PREPERATION
#===============================================

# 1. LOADING PARQUET FILE 

hand_files = sorted(glob.glob("../parquet/hands/**/*.parquet",recursive = True))
decisions_files = sorted(glob.glob("../parquet/decisions/**/*.parquet", recursive = True))

#hands_df = pd.read_parquet("../parquet/hands",engine = "fastparquet").head(2000)
#decisions_df = pd.read_parquet("../parquet/decisions", engine = "fastparquet").head(5000)

hands_df = pd.concat([pd.read_parquet(f,engine = "fastparquet") for f in hand_files[:5]], ignore_index=True)
decisions_df = pd.concat([pd.read_parquet(f,engine = "fastparquet") for f in decisions_files[:5]], ignore_index=True)


#merging into single dataframe
features_df = pd.merge(hands_df,decisions_df,on="hand_id",how="inner")

#mapping from parquet file into required feature colum
features_df["high_card_rank"] = np.maximum(features_df["hole1"] % 13, features_df["hole2"] % 13)
features_df["is_pair"] = ((features_df["hole1"] % 13) == (features_df["hole2"] % 13)).astype(int)
features_df["is_suited"] = ((features_df["hole1"] // 13) == (features_df["hole2"] // 13)).astype(int)
features_df["starting_hand"] = features_df["high_card_rank"] * 13 + np.minimum(features_df["hole1"] % 13, features_df["hole2"] % 13)
features_df["position_id"] = features_df["seat"]
features_df["raises_so_far"] = features_df["n_raises"]
features_df["amount_to_call"] = features_df["to_call"]
features_df["pot_size"] = features_df["pot"]
features_df["stack_bb"] = features_df["stack"] / features_df["big_blind"]

#selecting the specific features
feature_columns = ["starting_hand","is_suited","is_pair","high_card_rank","position_id","raises_so_far","amount_to_call","pot_size","stack_bb"]

print("\n------- First 10 columns of the dataset----------\n")
preview_cols = ["hand_id","starting_hand","is_suited","is_pair","high_card_rank","position_id","raises_so_far","amount_to_call"]
print(features_df[preview_cols].head(10))
print("\n--------------------------------------------------\n")

#Extract the array
train_samples = features_df[feature_columns].to_numpy()
train_samples_labels = features_df['action'].to_numpy().flatten()



------- First 10 columns of the dataset----------

   hand_id  starting_hand  is_suited  is_pair  high_card_rank  position_id  \
0  1000000            -88          1        1              12            2   
1  1000000            -88          1        1              12            3   
2  1000000            -88          1        1              12            4   
3  1000000            -88          1        1              12            5   
4  1000000            -88          1        1              12            0   
5  1000000            -88          1        1              12            1   
6  1000001            -88          1        1              12            2   
7  1000001            -88          1        1              12            3   
8  1000001            -88          1        1              12            4   
9  1000001            -88          1        1              12            0   

   raises_so_far  amount_to_call  
0              0            10.0  
1              1   

In [8]:
#=================================================
#TRAINING THE MODEL
#===================================================
print ("Initializing multiclass Logistic regression model")

poker_model = softmaxlogisticreg(learning_rate=0.05,max_iter=300)

#Train using Gradient descent
poker_model.fit(train_samples,train_samples_labels)

#print final summary
final_acc = poker_model.score(train_samples,train_samples_labels)
print(f"\n Final Training accuracy :  {final_acc * 100:.2f}%")


Initializing multiclass Logistic regression model
iteration 0 | Cross Entropy loss 1.0916934255957575
iteration 50 | Cross Entropy loss 27.820235544233558
iteration 100 | Cross Entropy loss 22.312806079050212
iteration 150 | Cross Entropy loss 22.556880879353294
iteration 200 | Cross Entropy loss 17.371841752984807
iteration 250 | Cross Entropy loss 15.67084965055153

 Final Training accuracy :  34.28%
